# 예제 02. 입력 구간과 예측값 만들기
빅데이터프로그래밍 · 11주차

## 목표
- `[1, 2, 3] → 4` 형태로 데이터를 자른다
- 5주차 Dataset 클래스를 시계열에 맞게 쓴다
- 입력 shape이 (batch, 시점, 특성)인 이유를 안다

시계열을 학습에 쓰려면 **긴 줄을 짧은 조각으로 잘라야** 합니다.


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)


## 1. 손으로 잘라 보기
길이 3의 창을 한 칸씩 옮기며 다음 값을 정답으로 삼습니다.


In [ ]:
data = [1, 2, 3, 4, 5, 6, 7, 8]
SEQ = 3

for i in range(len(data) - SEQ):
    x = data[i:i+SEQ]
    y = data[i+SEQ]
    print(f"{x} → {y}")


길이 8의 데이터에서 창 3개짜리 표본 5개가 나옵니다. **len − SEQ** 개입니다.


## 2. Dataset 클래스로 — 5주차와 같은 세 메서드


In [ ]:
class SeriesDataset(Dataset):
    def __init__(self, series, seq_len):
        self.series = torch.tensor(series, dtype=torch.float32)
        self.seq_len = seq_len

    def __len__(self):
        return len(self.series) - self.seq_len

    def __getitem__(self, i):
        x = self.series[i : i + self.seq_len]          # (seq_len,)
        y = self.series[i + self.seq_len]              # 스칼라
        return x.unsqueeze(-1), y.unsqueeze(-1)       # (seq_len, 1), (1,)


t = np.arange(0, 200)
sine = np.sin(t * 0.1)

ds = SeriesDataset(sine, seq_len=20)
print("표본 수:", len(ds))

x0, y0 = ds[0]
print("x shape:", tuple(x0.shape), "→ (시점, 특성)")
print("y shape:", tuple(y0.shape))


`unsqueeze(-1)` 로 특성 축을 만듭니다. 값이 하나뿐이어도 축이 필요합니다.


## 3. DataLoader로 batch 만들기


In [ ]:
loader = DataLoader(ds, batch_size=16, shuffle=True)
xb, yb = next(iter(loader))

print("입력:", tuple(xb.shape), "→ (batch, 시점, 특성)")
print("정답:", tuple(yb.shape))


| 축 | 뜻 | 여기서는 |
| --- | --- | --- |
| 0 | batch | 16개 표본 |
| 1 | 시점 | 과거 20개 |
| 2 | 특성 | 값 1개 |

이미지의 (batch, 채널, 높이, 너비)와 같은 방식입니다. 축의 이름만 다릅니다.


## 4. 잘라낸 표본을 그려 보기


In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(15, 2.8))
for ax, i in zip(axes, [0, 40, 80, 120]):
    x, y = ds[i]
    ax.plot(range(20), x.squeeze(), marker="o", markersize=3, label="입력 20개")
    ax.plot(20, y.item(), marker="*", markersize=14, color="crimson", label="정답")
    ax.set_title(f"표본 {i}"); ax.grid(alpha=.3)
    if i == 0: ax.legend(fontsize=8)
plt.tight_layout(); plt.show()


## 5. 입력 구간 길이를 바꾸면


In [ ]:
import pandas as pd

rows = []
for seq in [5, 10, 20, 50]:
    d = SeriesDataset(sine, seq_len=seq)
    rows.append({"seq_len": seq, "표본 수": len(d),
                 "한 표본 크기": tuple(d[0][0].shape)})
pd.DataFrame(rows)


구간을 길게 하면 더 많은 과거를 보지만 표본 수가 줄어듭니다. 과제에서 이 값을 바꿔 실험합니다.


## 6. 학습·검증 분할 — 시간 순서를 지킵니다


In [ ]:
n_train = int(len(sine) * 0.8)
train_ds = SeriesDataset(sine[:n_train], seq_len=20)
val_ds   = SeriesDataset(sine[n_train:], seq_len=20)

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=16, shuffle=False)

print(f"학습 표본 {len(train_ds)}개 · 검증 표본 {len(val_ds)}개")
print("검증은 shuffle=False — 순서대로 예측해 그려야 합니다")


## 7. 여러 값을 함께 넣기 — 특성이 2개 이상


In [ ]:
# 사인파와 코사인파를 함께 입력으로
two = np.stack([np.sin(t*0.1), np.cos(t*0.1)], axis=1)   # (200, 2)
print("원본:", two.shape)

class MultiDataset(Dataset):
    def __init__(self, arr, seq_len):
        self.arr = torch.tensor(arr, dtype=torch.float32)
        self.seq_len = seq_len
    def __len__(self):
        return len(self.arr) - self.seq_len
    def __getitem__(self, i):
        return self.arr[i:i+self.seq_len], self.arr[i+self.seq_len, 0:1]

md = MultiDataset(two, 20)
x, y = md[0]
print("x:", tuple(x.shape), "→ 특성 2개")
print("y:", tuple(y.shape))


## 직접 해보기
1. `seq_len=10` 으로 만들고 표본 수를 확인하세요.
2. 다음 **한 값**이 아니라 **다음 세 값**을 예측하도록 Dataset을 고치세요.
3. 표본 하나를 골라 입력과 정답을 그려 보세요.


In [ ]:
# 여기에 작성하세요
